<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/homework/WRIT20833_HW3_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3 — Sentiment Analysis: Support, Opposition, and What Counting Missed

**WRIT 20833 — Intro to Coding in the Humanities**

**Student Name:** _Replace with your name_

**Upload as:** `LASTNAME_HW3.ipynb`

---

## What you're doing
In **HW2** you found which *words* dominate a debate — but a raw count can't tell a **cheer**
from a **complaint**. The word *commandments* "won," yet some of those mentions were people
**for** the Texas law and some **against** it, and they looked identical.

This week you add **sentiment analysis**: a tool (VADER) that scores each comment from very
negative to very positive. You'll work in **pandas** now — a table where each row is one
comment — and use sentiment to hear the **stance** that counting alone missed.

## Prepare
Review **our own** materials first:
- **HW2** — term frequency (`split_into_words`, `stopwords`, `Counter`); we reuse all of it.
- **Pandas 01 / Pandas 02** code-alongs — DataFrames, `df[...]`, `.apply()`, `df.plot()`.
- The **VADER Sentiment** code-along (this week) — `analyzer.polarity_scores()` and the chart.

## Expected: frequent, meaningful `#comments`
Comment as you work — not every line, but **often, and wherever you make a choice or hit a
question**: say **what you tried** and **what you learned** (or what surprised you). Frequent,
relevant `#comments` are expected. Under this course's **ungrading** approach, your thinking in
these comments and reflections matters more than getting a "right" number.

## About the setup cell (read this, then just run it)
The next cell is **plumbing**: it imports pandas, installs the VADER sentiment tool, and loads
your data into a table. **Run it, but don't worry about reading every line** — installing a
library and downloading a file are things you'll meet later. You *use* a tool through its
**interface** long before you build its **insides**.

After it runs, you have: a DataFrame **`df`** (one comment per row), the name of its text column
**`text_column`**, the **`analyzer`** (VADER), and your HW2 **`split_into_words`** + **`stopwords`**.

**Bringing your own data?** The cell has a commented `pd.read_csv(...)` line — point it at the
cleaned CSV from the Day 10 workshop and set `text_column`. Otherwise it loads the course corpus
so everything runs out of the box.

In [ ]:
# SETUP — run this cell first. (You're not expected to read every line yet — that's fine.)
import re, os, urllib.request
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# VADER is a ready-made sentiment tool. It is not built into Python, so we install it,
# then make one analyzer we reuse. (More on "borrowing" tools in the note after A5.)
!pip install -q vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

# Same word-splitter + stopwords you built in HW2 (we reuse our own tools).
def split_into_words(any_chunk_of_text):
    return re.split(r"\W+", str(any_chunk_of_text).lower())
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll", "amp"]

def load_course_comments():
    """Load the course YouTube-comments corpus into a DataFrame, one comment per row."""
    filename = "tc_youtube_comments.txt"
    text = None
    for path in (filename, os.path.join("notebooks", "data", filename),
                 os.path.join("data", filename), os.path.join("..", "data", filename)):
        if os.path.exists(path):
            text = open(path, encoding="utf-8").read(); break
    if text is None:
        url = "https://raw.githubusercontent.com/TCU-DCDA/WRIT20833_2026/main/notebooks/data/" + filename
        text = urllib.request.urlopen(url).read().decode("utf-8")
    rows = [line.strip() for line in text.split("\n") if line.strip()]
    return pd.DataFrame({"comment": rows})

# ----- Choose your data -----------------------------------------------------------
# EXPECTED PATH -- bring your own: load the cleaned CSV you made in the Day 10 workshop.
#     df = pd.read_csv("YOUR_FILE.csv")
#     text_column = "the_text_column"      # e.g. "comment"
#
# FALLBACK (runs out of the box) -- the course corpus on the TX Ten Commandments law:
df = load_course_comments()
text_column = "comment"
# ---------------------------------------------------------------------------------

print(f"{len(df)} rows loaded. Text column: '{text_column}'.")
df.head()

## Part A — From text to a sentiment score (5 exercises)

**A1 — Meet your data as a table.** *(HW2 was one big blob; now each row is one comment.)*

In [ ]:
# A1
# TODO: print df.shape (rows, columns)
# TODO: print how many comments there are with len(df)
# TODO: show the first few rows with df.head()

# #comments:
# Each ROW is now one comment. How is a DataFrame different from HW2's flat word list?

**A2 — Clean text, but keep the feeling.** HW2 *dropped* punctuation and capitals — sentiment tools need them.

In [ ]:
# A2
sample = df[text_column].iloc[0]      # the first comment
print("Original:   ", sample)
# TODO: print sample.lower() and notice what intensity would be LOST (CAPS, "!") if we
#       cleaned it the HW2 way.

# #comments:
# Why does lowercasing + stripping "!" HELP a word count (HW2) but HURT a sentiment score?

**A3 — Score a single comment with VADER.**

In [ ]:
# A3
text = "This is a great day for religious freedom!"
scores = analyzer.polarity_scores(text)
print(scores)

# TODO: print just the 'compound' score (one number, -1 = most negative ... +1 = most positive)
# TODO: change `text` to something you might write, and re-run

# #comments:
# What do 'pos', 'neu', 'neg', and 'compound' each seem to measure?

**A4 — An edge case: predict, then run.**

In [ ]:
# A4 — predict, then run
# FIRST predict, in a comment: will VADER score this sarcastic line positive or negative?
# THEN run the cell.
sarcasm = "Oh great, another set of commandments to follow."
print(analyzer.polarity_scores(sarcasm)["compound"])

# #comments:
# My prediction was:
# VADER sees "great" and scores this POSITIVE -- but a human hears sarcasm.
# What does that warn you about trusting an automated score on its own?

**A5 — Score every comment at once.** *(This is what pandas is for.)*

In [ ]:
# A5
def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]

# TODO: make a new column df["sentiment"] by applying get_sentiment to df[text_column]
#       (hint: df[text_column].apply(get_sentiment))
# TODO: print df["sentiment"].mean(), .min(), .max()
# TODO: df.head() to see the new column

# #comments:
# In HW2 you looped by hand; here .apply() runs your function on every row.
# What just happened to all the comments in one line?

### A note: you didn't build VADER — and that's normal
VADER is a **sentiment lexicon**: thousands of words hand-scored for feeling, plus rules for
"!", ALL CAPS, and negation, compiled and tuned by researchers. Almost nobody writes one from
scratch — you `pip install` it. (Long before AI, you'd still have reached for a library like
this; today you might ask an AI to help use it.) **Borrowing it is expected, not cheating.**

But A4 showed the catch: a borrowed tool carries someone else's assumptions and can be
**confidently wrong** (it missed the sarcasm). So the real skill isn't *building* VADER — it's
**judging whether to trust it** on your text. That's exactly what the human-vs-VADER check in
**B3** is for. *(We pick this thread up on **Day 7**, reading and improving AI-written code.)*

## Part B — Support, opposition, and what counting missed (4 exercises)

**B1 — Turn scores into labels.** A number becomes a category. *(Recall Classification Logic.)*

In [ ]:
# B1
def label_sentiment(score):
    # Standard VADER cutoffs: >= 0.05 is positive, <= -0.05 is negative, otherwise neutral.
    # TODO: return "positive", "negative", or "neutral" based on score
    pass

# TODO: make df["label"] by applying label_sentiment to df["sentiment"]
# TODO: print df["label"].value_counts()

# #comments:
# WHO chose the 0.05 cutoff? How would moving it change how many comments count as "positive"?

**B2 — Picture the split.** One line of pandas makes a chart. *(You saw `df.plot()` in the code-alongs.)*

In [ ]:
# B2
# TODO: make a bar chart of the label counts:
#       df["label"].value_counts().plot(kind="bar", title="Sentiment of the comments")

# #comments:
# Does the chart show a clear majority, or a divided crowd?

**B3 — Read the extremes. Do you agree with VADER?** *(The human-vs-machine check.)*

In [ ]:
# B3
most_positive = df.loc[df["sentiment"].idxmax()]
most_negative = df.loc[df["sentiment"].idxmin()]
print("MOST POSITIVE (", round(most_positive["sentiment"], 3), "):")
print(most_positive[text_column])
print("\nMOST NEGATIVE (", round(most_negative["sentiment"], 3), "):")
print(most_negative[text_column])

# TODO: read BOTH comments yourself. Do YOU agree with VADER's call on each?

# #comments:
# Where do you AGREE with VADER, and where does it miss something a human catches?
# (Recall A4: sarcasm, irony, context.)

**B4 — Interpretation (write 4–6 sentences below).**

Using B1–B3: does this crowd mostly **support or oppose** the law — or is it split? Then the key
question: **what did sentiment let you see that HW2's word-counting could not?** In HW2 the word
*commandments* "won" by raw frequency. Cite at least two numbers from this notebook (a label
count, a sentiment score). Then name **one thing VADER still gets wrong** — a place the number and
your human reading disagree. *(Recall HW2 B4: a tally can't tell a cheer from a complaint.)*

_Replace this cell with your interpretation._

## Part C — Going deeper (2 exercises)

**C1 — Whose words are whose? Frequency *inside* each camp.** *(HW2's tool, now split by stance.)*

In [ ]:
# C1
def top_words(text_series, n=8):
    words = []
    for t in text_series:
        words.extend(w for w in split_into_words(t) if w and w not in stopwords)
    return Counter(words).most_common(n)

positive_comments = df[df["label"] == "positive"][text_column]
negative_comments = df[df["label"] == "negative"][text_column]

# TODO: print top_words(positive_comments) and top_words(negative_comments)

# #comments:
# Which words appear in BOTH lists? In HW2 those words "won" overall -- but frequency alone
# couldn't tell you they were used by BOTH sides. What does that mean for "what the data says"?

**C2 — On your own data, or push this one further.**

- **If you loaded your own dataset in setup:** this whole notebook already ran on it. Write 3–4
  sentences on what the sentiment split showed about *your* corpus, and one place you doubt VADER.
- **If you used the course corpus:** try ONE of — change the `0.05` cutoff in B1 and see what
  moves; or compare the sentiment of comments that mention `"constitution"` vs. those that don't.

In [ ]:
# C2
# TODO: do ONE of the options above. Show your code and the result.

# #comments:
# What did you expect, and did the data agree?

## Weekly Experiments (your own work!)
After the required exercises, create **2–3 small experiments** of your own with this week's
skills. Make them original.

Ideas:
- Write five comments yourself and see if VADER scores them the way you meant.
- Find a comment where VADER is **most wrong** and explain why.
- Filter `df` to a subset (e.g. comments mentioning "school") and compare its average sentiment.

It's fine if an experiment doesn't work at first — **documenting what went wrong and what you
learned is worth just as much as working code.**

In [ ]:
# Experiment 1


# #comments:
# What were you trying? What happened? What did you learn?

In [ ]:
# Experiment 2


# #comments:
# What were you trying? What happened? What did you learn?

In [ ]:
# Experiment 3


# #comments:
# What were you trying? What happened? What did you learn?

## Submit
- [ ] My name is at the top.
- [ ] I loaded a dataset (my own from the workshop, or the course corpus) and ran every cell. Where something still breaks, I left a `#comment` about what I tried and what I learned — errors are part of the work here, not something to hide or delete.
- [ ] I completed **Part A (5)**, **Part B (4, incl. the B4 write-up)**, **Part C (2)**, and **2–3 experiments**.
- [ ] I used `#comments` frequently and meaningfully — showing my thinking wherever I made a choice or hit a question (not every line, but throughout).
- [ ] I wrote honestly about where I **agree and disagree** with VADER (engagement matters more than "right" labels).
- [ ] Saved as `LASTNAME_HW3.ipynb` and uploaded to the D2L Dropbox.

**Time estimate:** ~2 hrs for the exercises, ~1 hr for your experiments.